## 🎯 Learning Objectives
* Implement a Corrective RAG system using LangGraph for advanced orchestration.
* Design and apply LangGraph's state management and conditional edges to build dynamic RAG workflows.
* Integrate Large Language Models (LLMs) for self-correction, answer critique, and query reformulation within a RAG pipeline.
* Understand and mitigate common RAG challenges like hallucination and irrelevant retrieval through iterative refinement.


## Exercise: Implement Corrective RAG with LangGraph

### Task
Your task is to build a Corrective RAG system using LangGraph. This system should be capable of self-correction: if an initial answer generated from retrieved documents is deemed unsatisfactory by an LLM-based critique, the system should reformulate the original query and attempt retrieval and generation again. This process should loop for a maximum number of attempts.

### Scenario
A user asks a question. Your system will:
1. **Retrieve** relevant documents based on the current query.
2. **Generate** an initial answer using the retrieved documents and the query.
3. **Critique** the generated answer using an LLM. The critique should determine if the answer is satisfactory or if the query needs to be rephrased for better retrieval.
4. If the answer is unsatisfactory, **rephrase** the query using an LLM.
5. Loop back to step 1 with the rephrased query, up to a predefined maximum number of attempts.
6. If the answer is satisfactory or the maximum attempts are reached, the process concludes.

### Requirements
1.  **LangGraph Orchestration**: Use LangGraph to define the workflow, including nodes and conditional edges.
2.  **Graph State**: Define a `TypedDict` for the graph state that includes at least:
    *   `question`: The current query string.
    *   `documents`: A list of retrieved document contents.
    *   `answer`: The generated answer string.
    *   `feedback`: The critique feedback (e.g., "satisfactory", "needs_rephrasing").
    *   `retrieval_attempts`: An integer counter for retrieval attempts.
3.  **Nodes**: Implement the following nodes:
    *   `retrieve_node`: Fetches documents based on the current `question` in the state.
    *   `generate_node`: Generates an answer based on the `question` and `documents` in the state.
    *   `critique_node`: An LLM-based node that evaluates the `answer` and `question`, updating the `feedback` in the state.
    *   `rephrase_query_node`: An LLM-based node that reformulates the `question` if `feedback` indicates a need for rephrasing.
4.  **Conditional Edges**: Implement conditional logic to loop back from `critique_node` to `rephrase_query_node` (and then `retrieve_node`) if the answer is unsatisfactory and `retrieval_attempts` are below the maximum.
5.  **Max Attempts**: Implement a mechanism to limit the number of retrieval/rephrasing attempts to prevent infinite loops (e.g., `MAX_RETRIEVAL_ATTEMPTS = 3`).
6.  **LLM Integration**: Use `ChatOpenAI` (or a compatible LLM) for generation, critique, and rephrasing tasks.
7.  **Mock Data**: For simplicity, you can use a mock list of documents and a basic keyword-based retriever for the `retrieve_node`.

### Evaluation Criteria
*   **Correctness**: The LangGraph workflow correctly implements the Corrective RAG logic.
*   **Functionality**: The system demonstrates self-correction by rephrasing queries and re-attempting retrieval when the initial answer is poor.
*   **State Management**: Effective use of LangGraph's state to pass information between nodes.
*   **Code Quality**: Clean, readable, and well-commented Python code.
*   **Robustness**: Handles the maximum retrieval attempts gracefully.


In [ ]:
import os
from typing import List, Dict, Any, TypedDict
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# --- Configuration and Setup ---

# Set your OpenAI API key
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize LLM (using a cost-effective model for exercises)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# Mock Document Store
mock_documents = [
    "The capital of France is Paris. Paris is known for its Eiffel Tower.",
    "Germany's capital is Berlin. Berlin has a rich history and many museums.",
    "The Amazon rainforest is the largest tropical rainforest in the world, primarily located in Brazil.",
    "Brazil is the largest country in South America and is famous for its carnival.",
    "Artificial intelligence is a rapidly evolving field that focuses on creating intelligent machines.",
    "Machine learning is a subset of AI that enables systems to learn from data without explicit programming.",
    "LangGraph is a library for building stateful, multi-actor applications with LLMs, inspired by Language Agent Tree Search (LATS).",
    "Corrective RAG involves an iterative process where an LLM critiques its own output and refines the retrieval or generation steps.",
    "Self-RAG is an advanced RAG technique where the LLM generates its own retrieval queries and critiques its own answers."
]

# --- Define Graph State ---

class GraphState(TypedDict):
    """Represents the state of our graph."""
    question: str
    documents: List[str]
    answer: str
    feedback: str
    retrieval_attempts: int

# --- Helper Functions (Nodes) ---

def retrieve_documents(state: GraphState) -> GraphState:
    """Retrieves documents based on the current question."""
    print("---RETRIEVE DOCUMENTS---")
    question = state["question"]
    current_attempts = state.get("retrieval_attempts", 0)
    
    # Simulate retrieval: simple keyword matching
    retrieved_docs = [doc for doc in mock_documents if any(word.lower() in doc.lower() for word in question.split())]
    
    # If no specific docs found, return a general set or all docs
    if not retrieved_docs:
        retrieved_docs = [doc for doc in mock_documents if len(doc.split()) > 5] # Return some general docs
    
    print(f"Retrieved {len(retrieved_docs)} documents for query: '{question}'")
    return {
        "documents": retrieved_docs,
        "retrieval_attempts": current_attempts + 1
    }

def generate_answer(state: GraphState) -> GraphState:
    """Generates an answer based on the question and retrieved documents."""
    print("---GENERATE ANSWER---")
    question = state["question"]
    documents = state["documents"]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert AI assistant. Answer the user's question based *only* on the provided context. If you cannot find the answer, state that you don't know."),
        ("user", "Context: {context}\n\nQuestion: {question}")
    ])
    
    rag_chain = prompt | llm | StrOutputParser()
    
    answer = rag_chain.invoke({"context": "\n".join(documents), "question": question})
    print(f"Generated answer: {answer[:100]}...")
    return {"answer": answer}

def critique_answer(state: GraphState) -> GraphState:
    """Critiques the generated answer and provides feedback."""
    print("---CRITIQUE ANSWER---")
    question = state["question"]
    answer = state["answer"]
    documents = state["documents"]
    
    critique_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert AI assistant tasked with evaluating answers. Your goal is to determine if the provided answer is satisfactory given the question and context. If the answer is directly supported by the context and fully addresses the question, respond with 'satisfactory'. If the answer is incomplete, incorrect, or not fully supported by the context, respond with 'needs_rephrasing' to indicate that the original question might need to be rephrased for better retrieval."),
        ("user", "Question: {question}\nContext: {context}\nAnswer: {answer}\n\nIs this answer satisfactory or does the question need rephrasing? Respond with 'satisfactory' or 'needs_rephrasing'.")
    ])
    
    critique_chain = critique_prompt | llm | StrOutputParser()
    
    feedback = critique_chain.invoke({"question": question, "context": "\n".join(documents), "answer": answer}).strip().lower()
    
    # Simple parsing to ensure consistent feedback
    if "satisfactory" in feedback:
        feedback = "satisfactory"
    else:
        feedback = "needs_rephrasing"
        
    print(f"Critique feedback: {feedback}")
    return {"feedback": feedback}

def rephrase_query(state: GraphState) -> GraphState:
    """Rephrases the original question for better retrieval."""
    print("---REPHRASE QUERY---")
    question = state["question"]
    answer = state["answer"]
    feedback = state["feedback"]
    
    rephrase_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert query rephrasing assistant. Given a question and an unsatisfactory answer, rephrase the original question to improve retrieval and get a better answer. Focus on clarifying ambiguities or adding more specific keywords. Return only the rephrased question."),
        ("user", "Original Question: {question}\nUnsatisfactory Answer: {answer}\nFeedback: {feedback}\n\nRephrased Question:")
    ])
    
    rephrase_chain = rephrase_prompt | llm | StrOutputParser()
    
    rephrased_question = rephrase_chain.invoke({"question": question, "answer": answer, "feedback": feedback})
    print(f"Rephrased question: {rephrased_question}")
    return {"question": rephrased_question}

# --- Conditional Edge Logic ---

MAX_RETRIEVAL_ATTEMPTS = 3

def decide_to_continue(state: GraphState) -> str:
    """Determines whether to continue the RAG process or end."""
    print("---DECIDE TO CONTINUE---")
    feedback = state["feedback"]
    retrieval_attempts = state["retrieval_attempts"]
    
    if feedback == "satisfactory":
        print("Decision: Answer is satisfactory. Ending process.")
        return "end"
    elif retrieval_attempts >= MAX_RETRIEVAL_ATTEMPTS:
        print(f"Decision: Max retrieval attempts ({MAX_RETRIEVAL_ATTEMPTS}) reached. Ending process.")
        return "end"
    else:
        print("Decision: Answer needs rephrasing. Looping for retry.")
        return "rephrase_and_retry"


### Your Implementation

Now, it's your turn to assemble these components into a LangGraph workflow. 

1.  **Instantiate `StateGraph`** with the `GraphState`.
2.  **Add Nodes**: Use the helper functions provided above as nodes.
3.  **Set Entry Point**: Define where the graph execution begins.
4.  **Add Edges**: Connect the nodes sequentially.
5.  **Add Conditional Edges**: Implement the `decide_to_continue` function to manage the loop for rephrasing and retrying.
6.  **Compile the Graph**: Create a runnable `CompiledGraph`.
7.  **Run the Graph**: Test your implementation with a few example questions, including one that might require rephrasing.


In [ ]:
from langgraph.graph import StateGraph, END

# --- Build the LangGraph Workflow ---

# 1. Instantiate StateGraph
workflow = StateGraph(GraphState)

# 2. Add Nodes
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("generate", generate_answer)
workflow.add_node("critique", critique_answer)
workflow.add_node("rephrase_query", rephrase_query)

# 3. Set Entry Point
workflow.set_entry_point("retrieve")

# 4. Add Edges
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "critique")
workflow.add_edge("rephrase_query", "retrieve") # Loop back to retrieval after rephrasing

# 5. Add Conditional Edges
workflow.add_conditional_edges(
    "critique", # From the critique node
    decide_to_continue, # Use the decision function
    {
        "rephrase_and_retry": "rephrase_query", # If needs rephrasing, go to rephrase_query
        "end": END # If satisfactory or max attempts, end the graph
    }
)

# 6. Compile the Graph
app = workflow.compile()

# --- Run the Graph with Example Questions ---

print("\n--- Running Corrective RAG System ---")

# Example 1: A straightforward question
print("\n--- Question 1: What is the capital of France? ---")
initial_state_1 = {
    "question": "What is the capital of France?",
    "documents": [],
    "answer": "",
    "feedback": "",
    "retrieval_attempts": 0
}
final_state_1 = app.invoke(initial_state_1)
print(f"Final Answer 1: {final_state_1['answer']}")
print(f"Attempts 1: {final_state_1['retrieval_attempts']}")
print(f"Feedback 1: {final_state_1['feedback']}")

# Example 2: A question that might need rephrasing due to ambiguity or lack of direct keywords
print("\n--- Question 2: Tell me about the big forest in South America. ---")
initial_state_2 = {
    "question": "Tell me about the big forest in South America.",
    "documents": [],
    "answer": "",
    "feedback": "",
    "retrieval_attempts": 0
}
final_state_2 = app.invoke(initial_state_2)
print(f"Final Answer 2: {final_state_2['answer']}")
print(f"Attempts 2: {final_state_2['retrieval_attempts']}")
print(f"Feedback 2: {final_state_2['feedback']}")

# Example 3: A question designed to hit max attempts (if critique is consistently 'needs_rephrasing')
# Note: The mock retriever and LLM might make it hard to consistently hit max attempts without fine-tuning prompts.
# This example aims to show the attempt counter working.
print("\n--- Question 3: What is the main idea behind self-learning AI systems? ---")
initial_state_3 = {
    "question": "What is the main idea behind self-learning AI systems?",
    "documents": [],
    "answer": "",
    "feedback": "",
    "retrieval_attempts": 0
}
final_state_3 = app.invoke(initial_state_3)
print(f"Final Answer 3: {final_state_3['answer']}")
print(f"Attempts 3: {final_state_3['retrieval_attempts']}")
print(f"Feedback 3: {final_state_3['feedback']}")
